# CamAiSearch en Google Colab

Este notebook instala dependencias, levanta la API y expone el dashboard completo mediante un tunel publico de Cloudflare (TryCloudflare).

In [ ]:
import os
import subprocess
from pathlib import Path

repo_dir = Path("/content/CamAiSearch")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/CarlosGTI001/CamAiSearch.git", str(repo_dir)], check=True)

os.chdir(repo_dir)
print("Working directory:", Path.cwd())

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU activa:", torch.cuda.get_device_name(0))
else:
    print("GPU no disponible. Cambia Runtime > Change runtime type > GPU.")

In [ ]:
from pathlib import Path
from google.colab import files

Path("videos").mkdir(parents=True, exist_ok=True)
uploaded = files.upload()

for filename, content in uploaded.items():
    output = Path("videos") / filename
    output.write_bytes(content)
    print("Saved:", output)

In [ ]:
import subprocess
import sys
import time

if "api_proc" in globals() and api_proc.poll() is None:
    api_proc.terminate()
    api_proc.wait(timeout=10)

cmd = [
    sys.executable,
    "main.py",
    "--config",
    "config/config.colab.json",
    "--host",
    "0.0.0.0",
    "--port",
    "8000",
]

api_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(8)
if api_proc.poll() is not None:
    raise RuntimeError("La API termino al arrancar. Revisa config/config.colab.json.")
print("API PID:", api_proc.pid)

In [ ]:
import json
import urllib.request

with urllib.request.urlopen("http://127.0.0.1:8000/events") as response:
    payload = json.loads(response.read().decode("utf-8"))

print("Events loaded:", len(payload))

In [ ]:
import re
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

cloudflared_path = Path("cloudflared")
if not cloudflared_path.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path,
    )
    cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IEXEC)

if "cloudflared_proc" in globals() and cloudflared_proc.poll() is None:
    cloudflared_proc.terminate()
    cloudflared_proc.wait(timeout=10)

log_path = Path("/tmp/cloudflared.log")
if log_path.exists():
    log_path.unlink()

cloudflared_cmd = [
    str(cloudflared_path),
    "tunnel",
    "--url",
    "http://127.0.0.1:8000",
    "--no-autoupdate",
    "--logfile",
    str(log_path),
    "--loglevel",
    "info",
]
cloudflared_proc = subprocess.Popen(cloudflared_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

public_url = None
deadline = time.time() + 60
pattern = re.compile(r"https://[-a-zA-Z0-9]+\\.trycloudflare\\.com")

while time.time() < deadline and cloudflared_proc.poll() is None:
    if log_path.exists():
        text = log_path.read_text(encoding="utf-8", errors="ignore")
        match = pattern.search(text)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)

if not public_url:
    raise RuntimeError("No se pudo obtener la URL publica del tunel.")

print("Dashboard publico:", public_url)
print("Eventos:", f"{public_url}/events")
print("WebSocket:", f"{public_url}/ws/live-events")

In [ ]:
import json
import urllib.request

body = json.dumps({
    "source": "videos/tu_video.mp4",
    "camera_id": "colab_demo",
    "max_seconds": 60,
    "save_clips": True
}).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:8000/analyze",
    data=body,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request) as response:
    print(response.read().decode("utf-8"))

In [ ]:
import subprocess

for proc_name in ("cloudflared_proc", "api_proc"):
    proc = globals().get(proc_name)
    if proc is None:
        continue
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"{proc_name} detenido.")
    else:
        print(f"{proc_name} ya estaba detenido.")